In [14]:
!pip install dotenv openai pandas openpyxl requests

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [requests]


In [1]:
from agno.agent import Agent, RunOutput
#from agno.models.anthropic import Claude
from agno.db.sqlite import SqliteDb
from agno.models.message import Message
from agno.run import RunContext
from agno.memory import MemoryManager
from agno.tools.serper import SerperTools

from agno.models.azure import AzureOpenAI

import os
import pandas as pd
from dotenv import load_dotenv  
from os import getenv
from pathlib import Path  



In [2]:
load_dotenv()
#os.environ["ANTHROPIC_API_KEY"] = getenv("ANTHROPIC_API_KEY")
os.environ["AZURE_OPENAI_API_KEY"] = getenv("AZURE_OPENAI_API_KEY")
os.environ["AZURE_ENDPOINT"] = getenv("AZURE_INFERENCE_ENDPOINT")  
os.environ["AZURE_OPENAI_DEPLOYMENT"] = getenv("LLM_MODEL")

In [3]:
#model = Claude(id="claude-sonnet-4-6")
model = AzureOpenAI(id=getenv("LLM-MODEL"), 
                    api_version=getenv("LLM_MODEL_VERSION"),
                    azure_endpoint=getenv("AZURE_INFERENCE_ENDPOINT"))
response = model.response(
    messages=[
        Message(role="user", content="Hello")
    ]
)
print(response.content)

Hello! How can I assist you today?


In [4]:
# ─── USER CONFIGURATION ──────────────────────────────────────────────────────
DATA_FOLDER   = Path('../data/input')   # <-- change to your folder

USE_CASE_FILE = DATA_FOLDER / 'use_case_Movie_Recommendation_v2_with_fi.xlsx'
ABT_FILE      = DATA_FOLDER / 'movie_abt_enriched_FINAL.xlsx'
TAXONOMY_FILE = DATA_FOLDER / 'Movie_Recommendation_TaxonomyV2.xlsx'
OSCARS_PDF    = DATA_FOLDER / 'The_98th_Academy_Awards_2026.pdf'

# SQLite file for Agno session memory
MEMORY_DB     = DATA_FOLDER / 'movie_agent_memory.db'
agent_db = SqliteDb(db_file=MEMORY_DB)


AZURE_OPENAI_API_KEY=os.getenv('AZURE_OPENAI_API_KEY')
AZURE_OPENAI_ENDPOINT=os.getenv('AZURE_OPENAI_ENDPOINT')
AZURE_OPENAI_API_VERSION=os.getenv("LLM_MODEL_VERSION") 
TMDB_API_KEY   = os.getenv('TMDB_API_KEY')
SERPER_API_KEY = os.getenv('SERPER_API_KEY')

LLM_MODEL        = os.getenv("LLM_MODEL")
#EMBED_MODEL      = 'text-embedding-3-small'
MAX_ABT_RESULTS  = 5
MAX_TAG_RESULTS  = 10
RAG_TOP_K        = 4
SESSION_ID       = 'movie-bot-session-001'  # change per user

print(f'Data folder: {DATA_FOLDER.resolve()}')
for f in [USE_CASE_FILE, ABT_FILE, TAXONOMY_FILE, OSCARS_PDF]:
    status = '✅' if f.exists() else '❌ NOT FOUND'
    print(f'  {status}  {f.name}')

Data folder: /Users/nsathi/Library/CloudStorage/Dropbox/Applied AI Institute/Courses/GenerativeAI/G-AI-5-Building-Full-Scale-AI-Solutions/AI-Sandbox/Agno-Class-exercises/data/input
  ✅  use_case_Movie_Recommendation_v2_with_fi.xlsx
  ✅  movie_abt_enriched_FINAL.xlsx
  ✅  Movie_Recommendation_TaxonomyV2.xlsx
  ✅  The_98th_Academy_Awards_2026.pdf


In [11]:
"""
Persistent Session
=============================

Persistent Session Example.
"""



# ---------------------------------------------------------------------------
# Create Agent
# ---------------------------------------------------------------------------
agent = Agent(
    model=model,
    db=agent_db,
    session_id="session_storage",
    add_history_to_context=True,
)

# ---------------------------------------------------------------------------
# Run Agent
# ---------------------------------------------------------------------------
while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)
    chat_history = agent.get_chat_history()
    for m in chat_history:
        print (m.content)

Your input:  I would like to watch a movie with my spouse.  I like romantic movie and he likes comedy

Agent:
How about trying a romantic comedy? It’s a great blend of both romance and humor, so you both might enjoy it! Here are a few popular romantic comedies you could consider:

1. **Crazy Rich Asians** – A funny and heartwarming story about love and family.
2. **The Proposal** – A romantic comedy with plenty of laughs starring Sandra Bullock and Ryan Reynolds.
3. **When Harry Met Sally...** – A classic romantic comedy that explores friendship and love.
4. **10 Things I Hate About You** – A modern, witty take on Shakespeare’s play with lots of laughs and romance.
5. **Notting Hill** – A charming story about a bookstore owner and a famous actress falling in love.

Would you like recommendations for where to stream these or something a bit different?
Tell me a new interesting fact about space
Here's an interesting new fact about space: 

Scientists recently discovered that some black h

In [10]:
chat_history = agent.get_chat_history()
for m in chat_history:
    print (m.content)

Tell me a new interesting fact about space
Here's an interesting new fact about space: 

Scientists recently discovered that some black holes can "sing" by emitting gravitational waves at specific frequencies, creating ripples in spacetime much like a musical instrument produces sound. These gravitational wave "tones" provide unique insights into the properties of black holes, such as their spin and mass, and open up new ways to study these mysterious cosmic objects beyond traditional electromagnetic observations. This phenomenon helps physicists test Einstein’s theory of general relativity in extreme environments with unprecedented precision.


In [5]:
df_instructions = pd.read_excel(USE_CASE_FILE, sheet_name='Agent Instructions')
AGENT_SYSTEM_PROMPT = df_instructions[df_instructions['Prompt Type'] == 'agent_prompt'].values[0][1]
print('System prompt loaded:')
print(AGENT_SYSTEM_PROMPT)




System prompt loaded:
You are a friendly and knowledgeable movie expert. Your ONLY role is to help users discover and learn about movies. Do not answer questions unrelated to movies.

UNDERSTAND THE USER (Flipped Interaction)
Before recommending any movie, collect the user's needs through a brief, engaging conversation. Ask ONE leading question at a time across these dimensions (do not ask all at once):
  • Viewing context: Are you watching alone, with family, friends, or a date?
  • Audience: Who is the audience — adults, teens, young children, mixed group?
  • Genre mood: What genre or emotional mood are you in? (e.g., action, comedy, thriller, romance, drama, sci-fi, documentary)
  • Decade/era: Any preference for era or decade? (classic, 1980s–90s, modern, recent)
  • Country/language: Any preference for country of origin or spoken language?
  • Oscar/award interest: Are you interested in critically acclaimed or award-nominated films?
Stop asking when you have enough to make a pers

In [6]:
memory_manager = MemoryManager(
    model=model,
    db=agent_db,
    additional_instructions="""
    Capture the user's movie preferences, including who they are watching with and their movie genre, language preferences.
    """,
)

In [7]:
agent = Agent(
    model=model,
    tools=[SerperTools()],
    session_state={"preferences": []},
    db=SqliteDb(db_file="tmp/agents.db"),
    instructions=AGENT_SYSTEM_PROMPT + "\nCurrent state (preferences) is: {preferences}",
    markdown=True,
    debug_mode=False,
    memory_manager=memory_manager,
    enable_agentic_memory=True,
    add_datetime_to_context=True,
    add_history_to_context=True,
    num_history_runs=5
 #   show_tool_calls           = True,   # debug: show which tools fire
)


print("Movie Agent is running. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)

    # ✅ Get updated preferences from session
    updated_preferences = run.session_state.get("preferences", [])

    # ✅ Store updated preferences into episodic memory
    if updated_preferences:
        memory_manager.add_memory(
            content=f"User preferences updated: {updated_preferences}",
            metadata={"type": "movie_preferences"}
        )

    # ✅ Retrieve latest stored memory
    stored_memories = memory_manager.get_memories(limit=5)

    print("\nUpdated Preferences (Session State):")
    print(updated_preferences)

    print("\nEpisodic Memory (Last 5 Entries):")
    for m in stored_memories:
        print("-", m.content)

    # ✅ Print session state after each run
    print("\nSession State:")
    print(run.session_state)

    print("\n" + "-"*50 + "\n")
#run = agent.run("What is the story for the movie Toy Story in 20 words or less", stream=False)

Movie Agent is running. Type 'exit' to quit.

Your input:  I would like to watch a romantic movie with my sweetheart.  He is interested in a comedy

Agent:
Great! A romantic comedy sounds like a perfect choice for watching with your sweetheart.

To narrow down the options, do you have any preference for the era or decade of the movie? For example, would you like a classic romantic comedy, something from the 80s or 90s, or a more modern one?


AttributeError: 'MemoryManager' object has no attribute 'get_memories'

In [ ]:
agent = Agent(
    model=model,
    tools=[SerperTools()],
    instructions=AGENT_SYSTEM_PROMPT + "\nChat History: {preferences}",
    markdown=True,
    debug_mode=False,
    memory_manager=memory_manager,
    enable_agentic_memory=True,
    add_datetime_to_context=True,
    add_history_to_context=True,
    num_history_runs=5
 #   show_tool_calls           = True,   # debug: show which tools fire
)

In [10]:
agent.get_chat_history()

[Message(id='ee5825b7-6dbd-4c3a-af33-d0c75260d534', role='user', content='I would like to watch a romantic movie with my sweetheart.  He is interested in a comedy', compressed_content=None, name=None, tool_call_id=None, tool_calls=None, audio=None, images=None, videos=None, files=None, audio_output=None, image_output=None, video_output=None, file_output=None, redacted_reasoning_content=None, provider_data=None, citations=None, reasoning_content=None, tool_name=None, tool_args=None, tool_call_error=None, stop_after_tool_call=False, add_to_agent_memory=True, from_history=False, metrics=MessageMetrics(input_tokens=0, output_tokens=0, total_tokens=0, audio_input_tokens=0, audio_output_tokens=0, audio_total_tokens=0, cache_read_tokens=0, cache_write_tokens=0, reasoning_tokens=0, cost=None, timer=None, duration=None, time_to_first_token=None, provider_metrics=None), references=None, created_at=1772507209, temporary=False),
 Message(id='7c82ce08-b7e1-44ce-a4bf-a0d35aa4cbbf', role='assistant',

In [ ]:



print("Movie Agent is running. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")
    print("Your input: ", user_input)
    
    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    run = agent.run(user_input, stream=False)
    
    print("\nAgent:")
    print(run.content)

    # ✅ Get updated preferences from session
    updated_preferences = run.session_state.get("preferences", [])

    # ✅ Store updated preferences into episodic memory
    if updated_preferences:
        memory_manager.add_memory(
            content=f"User preferences updated: {updated_preferences}",
            metadata={"type": "movie_preferences"}
        )

    # ✅ Retrieve latest stored memory
    stored_memories = memory_manager.get_memories(limit=5)

    print("\nUpdated Preferences (Session State):")
    print(updated_preferences)

    print("\nEpisodic Memory (Last 5 Entries):")
    for m in stored_memories:
        print("-", m.content)

    # ✅ Print session state after each run
    print("\nSession State:")
    print(run.session_state)

    print("\n" + "-"*50 + "\n")

[]

In [18]:
print(run.content)

Toy Story: A cowboy doll feels threatened when a new spaceman toy arrives, leading to adventures and friendship.
